# IT3051 - Fundamentals of Data Mining & Machine Learning
## Mini Project 2026: Used Car Price Prediction System

**Assigned Model:** Decision Tree Regressor  
**Target Variable:** `log_price` (log-transformed vehicle selling price)  
**Input Features:** 13 engineered and scaled features from `Used_Car_Price_Prediction_System.ipynb`

In [1]:
# 1. Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 2. Load Preprocessed Datasets

In [ ]:
# Load preprocessed datasets
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape} (3,206 samples, 13 features)")
print(f"X_test  shape: {X_test.shape} (802 samples, 13 features)")

X_train shape: (3206, 13) (3,206 samples, 13 features)
X_test  shape: (802, 13) (802 samples, 13 features)


---
## 3. Baseline Decision Tree (Unconstrained Depth)
We train an unpruned Decision Tree Regressor to illustrate baseline performance and diagnose overfitting.

In [ ]:
# 5-fold CV setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
base_tree = DecisionTreeRegressor(random_state=42)

base_cv = cross_validate(
    base_tree,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

base_cv_r2   = base_cv['test_r2'].mean()
base_cv_r2_s = base_cv['test_r2'].std()
base_cv_mae  = -base_cv['test_mae'].mean()
base_cv_rmse = -base_cv['test_rmse'].mean()

# Fit and test baseline on untouched test set
base_tree.fit(X_train, y_train)
y_pred_base = base_tree.predict(X_test)

base_test_r2   = r2_score(y_test, y_pred_base)
base_test_mae  = mean_absolute_error(y_test, y_pred_base)
base_test_rmse = root_mean_squared_error(y_test, y_pred_base)

y_test_dollar = np.expm1(y_test)
y_pred_base_dollar = np.expm1(y_pred_base)
base_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_base_dollar)
base_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_base_dollar)

print("=== Baseline Decision Tree (Unpruned) Performance ===")
print(f"CV R²                 : {base_cv_r2:.4f} (+/- {base_cv_r2_s:.4f})")
print(f"CV MAE                : {base_cv_mae:.4f}")
print(f"CV RMSE               : {base_cv_rmse:.4f}")
print(f"Test R²               : {base_test_r2:.4f}")
print(f"Test MAE ($ scale)    : ${base_test_mae_dollar:,.2f}")
print(f"Test RMSE ($ scale)   : ${base_test_rmse_dollar:,.2f}")
print("\nObservation: The unpruned tree overfits heavily, yielding poor generalization (Test R² ~ 0.39).")

=== Baseline Decision Tree (Unpruned) Performance ===
CV R²                 : 0.3631 (+/- 0.0202)
CV MAE                : 0.5124
CV RMSE               : 0.6736
Test R²               : 0.3929
Test MAE ($ scale)    : $25,724.18
Test RMSE ($ scale)   : $79,266.97

Observation: The unpruned tree overfits heavily, yielding poor generalization (Test R² ~ 0.39).


---
## 4. Hyperparameter Tuning (Stage 7: Optimization)
We use `GridSearchCV` on the training set to prune the tree using `max_depth`, `min_samples_leaf`, `min_samples_split`, and `max_features`.

In [ ]:
# Hyperparameter search space
param_grid = {
    'max_depth': [4, 6, 8, 10, 12, None],
    'min_samples_split': [2, 10, 20, 30],
    'min_samples_leaf': [1, 5, 10, 15],
    'max_features': [None, 'sqrt', 0.8]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring='r2',#choose a hyperpara meter combination that produce highet r2
    cv=kf,
    n_jobs=-1 #Use all available CPU cores.
)
grid_search.fit(X_train, y_train) #first do only with training set

best_params = grid_search.best_params_
print("=== Best Decision Tree Hyperparameters ===")
print(f"Best Parameters: {best_params}")
print(f"Best CV R²     : {grid_search.best_score_:.4f}")

=== Best Decision Tree Hyperparameters ===
Best Parameters: {'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 15, 'min_samples_split': 2}
Best CV R²     : 0.6303


---
## 5. Tuned Decision Tree Evaluation & Comparison

In [ ]:
# Evaluate Tuned Decision Tree
tuned_tree = DecisionTreeRegressor(**best_params, random_state=42)

tuned_cv = cross_validate(
    tuned_tree,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

tuned_cv_r2   = tuned_cv['test_r2'].mean()
tuned_cv_r2_s = tuned_cv['test_r2'].std()
tuned_cv_mae  = -tuned_cv['test_mae'].mean()
tuned_cv_rmse = -tuned_cv['test_rmse'].mean()

tuned_tree.fit(X_train, y_train)
y_pred_tuned = tuned_tree.predict(X_test)

tuned_test_r2   = r2_score(y_test, y_pred_tuned)
tuned_test_mae  = mean_absolute_error(y_test, y_pred_tuned)
tuned_test_rmse = root_mean_squared_error(y_test, y_pred_tuned)

y_pred_tuned_dollar = np.expm1(y_pred_tuned)
tuned_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_tuned_dollar)
tuned_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_tuned_dollar)

print("=== Tuned Decision Tree Performance ===")
print(f"Tuned CV R²           : {tuned_cv_r2:.4f} (+/- {tuned_cv_r2_s:.4f})")
print(f"Tuned CV MAE          : {tuned_cv_mae:.4f}")
print(f"Tuned CV RMSE         : {tuned_cv_rmse:.4f}")
print(f"Tuned Test R²         : {tuned_test_r2:.4f}")
print(f"Tuned Test MAE ($)    : ${tuned_test_mae_dollar:,.2f}")
print(f"Tuned Test RMSE ($)   : ${tuned_test_rmse_dollar:,.2f}")

error_reduction = base_test_mae_dollar - tuned_test_mae_dollar
print(f"\nPruning Impact: Reduced Dollar MAE by ${error_reduction:,.2f} and increased R² by {(tuned_test_r2 - base_test_r2)*100:.1f}%!")

=== Tuned Decision Tree Performance ===
Tuned CV R²           : 0.6303 (+/- 0.0180)
Tuned CV MAE          : 0.3942
Tuned CV RMSE         : 0.5129
Tuned Test R²         : 0.6299
Tuned Test MAE ($)    : $18,410.47
Tuned Test RMSE ($)   : $37,918.03

Pruning Impact: Reduced Dollar MAE by $7,313.71 and increased R² by 23.7%!


---
## 6. Feature Importances & Saving Results

In [ ]:
# Feature importances
os.makedirs(os.path.join('results', 'feature_importance'), exist_ok=True)
dt_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': tuned_tree.feature_importances_
}).sort_values('Importance', ascending=False)
dt_imp.to_csv(os.path.join('results', 'feature_importance', 'decision_tree_importance.csv'), index=False)

# Save results records
records = [
    {
        'Model': 'Decision Tree',
        'Version': 'Baseline',
        'Parameters': 'max_depth=None (unpruned)',
        'CV_R2': base_cv_r2,
        'CV_R2_Std': base_cv_r2_s,
        'CV_MAE': base_cv_mae,
        'CV_RMSE': base_cv_rmse,
        'Test_R2': base_test_r2,
        'Test_MAE': base_test_mae,
        'Test_RMSE': base_test_rmse,
        'Test_MAE_Dollar': base_test_mae_dollar,
        'Test_RMSE_Dollar': base_test_rmse_dollar,
        'Observation': 'Severe overfitting observed due to unconstrained leaf growth.'
    },
    {
        'Model': 'Decision Tree',
        'Version': 'Tuned',
        'Parameters': str(best_params),
        'CV_R2': tuned_cv_r2,
        'CV_R2_Std': tuned_cv_r2_s,
        'CV_MAE': tuned_cv_mae,
        'CV_RMSE': tuned_cv_rmse,
        'Test_R2': tuned_test_r2,
        'Test_MAE': tuned_test_mae,
        'Test_RMSE': tuned_test_rmse,
        'Test_MAE_Dollar': tuned_test_mae_dollar,
        'Test_RMSE_Dollar': tuned_test_rmse_dollar,
        'Observation': 'Pruning with ' + str(best_params) + ' regularized the tree and recovered generalization.'
    }
]

pd.DataFrame(records).to_csv(os.path.join('results', 'decision_tree_results.csv'), index=False)
print("Decision Tree evaluation saved to results/decision_tree_results.csv")

Decision Tree evaluation saved to results/decision_tree_results.csv
